# Lab 2：昇腾环境检查与连通验证

## 实验目标

- 识别 Notebook 使用的 Python Kernel 和 Host 基本信息。
- 查看驱动识别到的 NPU、设备状态以及 CANN 安装信息。
- 理解 ACL、`torch_npu` 与 CANN Runtime 之间的关系。
- 使用最小 NPU 张量检查框架到设备的连通性。

## 实验环境

| 项目 | 配置 |
| --- | --- |
| NPU | 单卡 Ascend 910B3（Atlas A2） |
| CANN | 9.0.0 |
| Python | 3.11.4 |
| 关键工具 | Jupyter Notebook、`npu-smi`、ACL、PyTorch、`torch_npu` |

## 实验原理

昇腾环境由几个相互衔接的软件层组成。Notebook Kernel 决定当前 Python 解释器和可用的软件包；`npu-smi` 从驱动侧读取设备状态；CANN Toolkit、OPP 与 Runtime 提供开发和运行能力；ACL 和 `torch_npu` 分别提供底层运行时接口与 PyTorch 适配接口。

本实验按 `Notebook Kernel → 驱动 → CANN → ACL → torch_npu → NPU 张量` 的顺序检查各层。某一层出现异常时，可以结合前一层的结果缩小排查范围。

## 实验流程

### 1. 确认 Notebook 使用的 Kernel

先查看 Notebook 连接的 Python 环境。后续的软件包版本和 NPU 检查都以 `sys.executable` 显示的解释器为准。本单元还会读取 Host 的架构、CPU 数量和内存信息。

In [ ]:
import importlib.metadata as metadata
import importlib.util
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

lab2 = {}  # 后续单元格共享检查结果

def gib(value):
    return f"{value / 1024**3:.1f} GiB"

def host_memory():
    info = {}
    try:
        for line in Path('/proc/meminfo').read_text().splitlines():
            key, value = line.split(':', 1)
            info[key] = int(value.strip().split()[0]) * 1024
        return gib(info['MemTotal']), gib(info['MemAvailable'])
    except (OSError, KeyError, ValueError):
        return '无法读取', '无法读取'

mem_total, mem_available = host_memory()
lab2['host_arch'] = platform.machine()
lab2['python'] = platform.python_version()
lab2['python_executable'] = sys.executable
print(f"操作系统      : {platform.system()} {platform.release()}")
print(f"CPU 架构      : {lab2['host_arch']}")
print(f"Python 版本   : {lab2['python']}")
print(f"Kernel 解释器 : {sys.executable}")
print(f"当前工作目录  : {Path.cwd()}")
print(f"逻辑 CPU 数   : {os.cpu_count()}")
print(f"主机内存      : 总计 {mem_total}，当前可用 {mem_available}")

记录输出中的 Kernel 路径，环境排查通常从这里开始。课程镜像的 CPU 架构应显示为 `aarch64`。本单元中的 CPU 和内存信息来自 Host，NPU 的 AI Core 与 HBM 状态需要通过驱动工具查看。

### 2. 查看 NPU 和驱动状态

`npu-smi info` 显示驱动识别到的设备及其实时状态。本单元同时保存命令返回码和原始输出，供后续单元提取设备健康状态。

In [ ]:
npu_smi = shutil.which('npu-smi')
lab2['npu_smi_path'] = npu_smi
if npu_smi:
    result = subprocess.run([npu_smi, 'info'], text=True, capture_output=True, timeout=20)
    npu_smi_text = (result.stdout + result.stderr).strip()
    lab2['npu_smi_ok'] = result.returncode == 0
    lab2['npu_smi_output'] = npu_smi_text
    print(npu_smi_text)
    if result.returncode:
        print(f"[WARN] npu-smi 返回码：{result.returncode}")
else:
    lab2['npu_smi_ok'] = False
    lab2['npu_smi_output'] = ''
    print('[FAIL] 当前 PATH 中找不到 npu-smi。')

可以从输出中读取以下信息：

- `Name` 给出芯片名称，`Health = OK` 表示设备健康状态正常。
- `NPU` 和 `Chip` 是管理工具中的编号；后文的 `npu:0` 是当前进程使用的逻辑设备编号。
- `Power(W)`、`Temp(C)`、`AICore(%)` 和 `Memory-Usage` 反映查询时的负载。设备空闲时，AI Core 利用率通常为 0。
- 顶部的 Version 属于管理工具或驱动侧，CANN Toolkit 版本需要从安装信息中读取。

进程表和部分字段可能随驱动版本变化，判断设备状态时以当前输出为准。

In [ ]:
# 从本次实际输出提取少量稳定信号；
text = lab2.get('npu_smi_output', '')
health_values = re.findall(r'\|\s*\d+\s+[^|]+\|\s*(OK|Warning|Alarm|Fault|Abnormal)\b', text, re.I)
lab2['npu_health_values'] = health_values
lab2['npu_health_ok'] = bool(health_values) and all(v.upper() == 'OK' for v in health_values)
version_match = re.search(r'npu-smi\s+([^\s|]+).*?Version:\s*([^\s|]+)', text, re.S | re.I)
print('npu-smi/驱动侧版本:', version_match.groups() if version_match else '未从当前格式提取')
print('检测到的 Health 值:', health_values or '未从当前格式提取（最终报告将提示人工查看原始输出）')

下面读取 `npu-smi info -h` 的帮助信息。可用参数由驱动版本决定，排查命令时应参考当前环境的输出。

In [ ]:
if npu_smi:
    help_result = subprocess.run([npu_smi, 'info', '-h'], text=True, capture_output=True, timeout=10)
    print((help_result.stdout + help_result.stderr).strip()[:5000])
else:
    print('[SKIP] npu-smi 不存在，无法查看子命令帮助。')

### 3. 检查 CANN Toolkit 与环境变量

CANN 通过环境变量向当前进程提供 Toolkit、OPP 和动态库路径。下面读取课程相关变量，并从 `PATH` 与 `LD_LIBRARY_PATH` 中筛选 Ascend/CANN 路径。

`ASCEND_HOME_PATH` 和 `ASCEND_TOOLKIT_HOME` 通常指向 Toolkit 根目录，`ASCEND_OPP_PATH` 指向 OPP 算子包。

In [ ]:
allowed = ['ASCEND_HOME_PATH', 'ASCEND_TOOLKIT_HOME', 'ASCEND_OPP_PATH', 'ASCEND_AICPU_PATH']
for key in allowed:
    print(f"{key:<22} = {os.environ.get(key, '<未设置>')}")
for key in ['PATH', 'LD_LIBRARY_PATH']:
    parts = [p for p in os.environ.get(key, '').split(os.pathsep)
             if 'ascend' in p.lower() or 'cann' in p.lower()]
    print(f"{key} 中的 Ascend 片段：")
    print('  ' + ('\n  '.join(parts) if parts else '<未发现>'))
lab2['cann_env_loaded'] = bool(os.environ.get('ASCEND_HOME_PATH') or os.environ.get('ASCEND_TOOLKIT_HOME'))

这些变量一般在 Kernel 启动时加载。切换 Kernel 或调整 CANN 配置后，需要重启运行环境才能读取新值。

下一单元会在常见安装目录中查找安装信息，并读取包名、版本和架构。

In [ ]:
roots = []
for value in [os.environ.get('ASCEND_HOME_PATH'), os.environ.get('ASCEND_TOOLKIT_HOME'),
              '/usr/local/Ascend/ascend-toolkit/latest', '/usr/local/Ascend/cann']:
    if value:
        path = Path(value).expanduser()
        if path.exists() and path not in roots:
            roots.append(path)

def nearby_files(root, filename):
    candidates = [root / filename, root / 'aarch64-linux' / filename, root / 'opp' / filename]
    return [p for p in candidates if p.is_file()]

info_files = []
for root in roots:
    for name in ['ascend_toolkit_install.info', 'ascend_ops_install.info', 'version.info']:
        info_files.extend(nearby_files(root, name))
info_files = list(dict.fromkeys(info_files))

def read_info(path):
    data = {}
    try:
        for line in path.read_text(errors='replace').splitlines():
            if '=' in line:
                key, value = line.split('=', 1)
                if key.strip().lower() in {'package_name', 'version', 'innerversion', 'arch'}:
                    data[key.strip().lower()] = value.strip().strip('\"')
    except OSError as exc:
        data['error'] = str(exc)
    return data

parsed_infos = [(p, read_info(p)) for p in info_files]
for path, data in parsed_infos:
    print(path)
    print('  ' + ', '.join(f'{k}={v}' for k, v in data.items()))

toolkit_info = next((d for p, d in parsed_infos if p.name == 'ascend_toolkit_install.info'), {})
ops_info = next((d for p, d in parsed_infos if p.name == 'ascend_ops_install.info'), {})
lab2['cann_roots'] = [str(p) for p in roots]
lab2['cann_version'] = toolkit_info.get('version')
lab2['ops_version'] = ops_info.get('version')
lab2['install_info_found'] = bool(toolkit_info)
print('CANN Toolkit 版本结论:', lab2['cann_version'] or '未检测到')
print('芯片算子包版本结论:', lab2['ops_version'] or '未检测到')

课程环境中的 CANN Toolkit 版本应为 9.0.0。版本以 `ascend_toolkit_install.info` 等安装文件中的字段为准，OPP 的版本和路径也应一并核对。

### 4. 查看 CANN 组件与常用工具

下面定位 OPP、Runtime、头文件、动态库和编译工具。输出中的“核心”项组成基础开发链路；图编译、模型转换、性能分析、多卡通信和推理服务等组件按需安装。

In [ ]:
cann_root_paths = [Path(p) for p in lab2.get('cann_roots', [])]
opp_env = os.environ.get('ASCEND_OPP_PATH')
opp_candidates = ([Path(opp_env)] if opp_env else []) + [r / 'opp' for r in cann_root_paths]
opp_path = next((p for p in opp_candidates if p.is_dir()), None)
include_path = next((p for r in cann_root_paths for p in [r/'include', r/'aarch64-linux'/'include'] if p.is_dir()), None)
lib64_path = next((p for r in cann_root_paths for p in [r/'lib64', r/'aarch64-linux'/'lib64', r/'runtime'/'lib64'] if p.is_dir()), None)
ascendc_path = next((p for r in cann_root_paths for p in [r/'aarch64-linux'/'ascendc', r/'include'/'ascendc', r/'aarch64-linux'/'include'/'ascendc'] if p.exists()), None)
set_env_path = next((r/'set_env.sh' for r in cann_root_paths if (r/'set_env.sh').is_file()), None)

def find_named(patterns):
    for root in cann_root_paths:
        for base in [root/'aarch64-linux'/'lib64', root/'lib64', root/'runtime'/'lib64']:
            if base.is_dir():
                for pattern in patterns:
                    match = next(base.glob(pattern), None)
                    if match:
                        return match
    return None

runtime_lib = find_named(['libascendcl.so*', 'libruntime.so*'])
hccl_lib = find_named(['libhccl.so*'])
ge_lib = find_named(['libge_compiler.so*', 'libge_runner.so*'])
tools = {name: shutil.which(name) for name in ['ccec', 'bisheng', 'atc', 'msprof', 'msame']}
mindie_candidates = [p for root in cann_root_paths for p in [root/'mindie', root/'MindIE']]
mindie_path = next((p for p in mindie_candidates if p.exists()), None)

lab2.update({
    'opp_path': str(opp_path) if opp_path else None,
    'include_path': str(include_path) if include_path else None,
    'lib64_path': str(lib64_path) if lib64_path else None,
    'ascendc_path': str(ascendc_path) if ascendc_path else None,
    'set_env_path': str(set_env_path) if set_env_path else None,
    'runtime_lib': str(runtime_lib) if runtime_lib else None,
    'hccl_lib': str(hccl_lib) if hccl_lib else None,
    'ge_lib': str(ge_lib) if ge_lib else None,
    'tools': tools, 'mindie_path': str(mindie_path) if mindie_path else None,
})

rows = [
    ('CANN ops/OPP', bool(opp_path), opp_path, '内置算子与芯片适配数据', '核心'),
    ('ACL/Runtime 库', bool(runtime_lib), runtime_lib, '设备、Stream、内存与执行接口', '核心'),
    ('CANN 头文件', bool(include_path), include_path, 'C/C++ 开发接口', '核心'),
    ('CANN lib64', bool(lib64_path), lib64_path, '运行与开发动态库', '核心'),
    ('Ascend C 目录', bool(ascendc_path), ascendc_path, '自定义算子开发', '核心'),
    ('CCEC/BiSheng', bool(tools['ccec'] or tools['bisheng']), tools['ccec'] or tools['bisheng'], 'Host/Device 编译工具', '核心'),
    ('ATC', bool(tools['atc']), tools['atc'], '模型转换为 OM', '可选'),
    ('Profiler/msprof', bool(tools['msprof']), tools['msprof'], '性能数据采集与分析', '可选'),
    ('HCCL', bool(hccl_lib), hccl_lib, '多卡集合通信', '可选'),
    ('GE', bool(ge_lib), ge_lib, '图优化、编译与执行', '可选'),
    ('msame', bool(tools['msame']), tools['msame'], 'OM 模型推理验证', '可选'),
    ('MindIE', bool(mindie_path), mindie_path, '推理服务', '可选'),
]
print(f"{'组件':<18} {'结果':<8} {'分类':<6} 路径/说明")
print('-' * 96)
for name, found, path, role, core in rows:
    status = 'PASS' if found and core == '核心' else ('OPTIONAL' if core == '可选' else 'WARN')
    print(f"{name:<18} {status:<8} {core:<6} {str(path) if path else '未检测到'}；{role}")

`PASS` 表示核心组件已定位，`WARN` 表示对应核心路径或工具未找到。`OPTIONAL` 只说明该组件属于扩展工具，不代表环境异常。

### 5. 进行 ACL Runtime 轻量检查

ACL（AscendCL）是 CANN 的运行时接口。下面检查 ACL Python binding，随后读取 SoC 名称和设备数量。代码只做轻量查询，不执行模型计算。

In [ ]:
lab2['acl_imported'] = False
lab2['acl_soc_name'] = None
lab2['acl_device_count'] = None
acl_spec = importlib.util.find_spec('acl')
if acl_spec is None:
    print('[WARN] 当前 Kernel 中没有发现 acl Python binding；不自动安装。')
else:
    try:
        # 当前 CANNLab 镜像中先加载 torch，可避免 acl.so 先加载后触发静态 TLS 空间冲突。
        # 这里只控制动态库加载顺序，不访问 NPU；版本与设备检查仍在下一节完成。
        if importlib.util.find_spec('torch') is not None and 'torch' not in sys.modules:
            import torch
        import acl
        lab2['acl_imported'] = True
        print('acl binding:', acl.__file__)
        try:
            lab2['acl_soc_name'] = acl.get_soc_name()
            print('acl.get_soc_name():', lab2['acl_soc_name'])
        except Exception as exc:
            print(f'[WARN] 读取 SoC 名称失败：{type(exc).__name__}: {exc}')
        try:
            count_result = acl.rt.get_device_count()
            # pyACL 常见返回值为 (count, ret_code)，也兼容直接返回整数的实现。
            lab2['acl_device_count'] = count_result[0] if isinstance(count_result, tuple) else count_result
            print('acl.rt.get_device_count():', count_result)
        except Exception as exc:
            print(f'[WARN] ACL 设备计数失败：{type(exc).__name__}: {exc}')
    except Exception as exc:
        print(f'[WARN] acl 导入失败：{type(exc).__name__}: {exc}')

如果 binding 路径、SoC 名称和设备数量能够正常输出，说明当前 Kernel 可以加载并调用 ACL 的 Python 接口。下一步从 PyTorch 访问同一设备。

### 6. 检查 PyTorch 与 torch_npu

`torch_npu` 把 PyTorch 的设备管理和算子调用接入昇腾 NPU。下面检查 PyTorch 与 `torch_npu` 的版本、导入状态以及框架可见的设备数量。

In [ ]:
lab2.update({'torch_imported': False, 'torch_npu_imported': False, 'npu_available': False,
             'torch_npu_device_count': 0, 'current_device': None, 'device_name': None})
try:
    import torch
    lab2['torch_imported'] = True
    lab2['torch_version'] = torch.__version__
    print('torch 版本:', lab2['torch_version'])
except Exception as exc:
    print(f'[FAIL] torch 导入失败：{type(exc).__name__}: {exc}')

try:
    lab2['torch_npu_dist_version'] = metadata.version('torch-npu')
except metadata.PackageNotFoundError:
    lab2['torch_npu_dist_version'] = None
print('torch-npu 发行包版本:', lab2['torch_npu_dist_version'] or '未检测到')

if lab2['torch_imported']:
    try:
        import torch_npu
        lab2['torch_npu_imported'] = True
        print('torch_npu 导入: 成功')
    except Exception as exc:
        print(f'[FAIL] torch_npu 导入失败：{type(exc).__name__}: {exc}')

if lab2['torch_npu_imported']:
    try:
        lab2['npu_available'] = bool(torch.npu.is_available())
        lab2['torch_npu_device_count'] = int(torch.npu.device_count())
        print('torch.npu.is_available():', lab2['npu_available'])
        print('torch.npu.device_count():', lab2['torch_npu_device_count'])
        if lab2['npu_available'] and lab2['torch_npu_device_count'] > 0:
            lab2['current_device'] = int(torch.npu.current_device())
            lab2['device_name'] = torch.npu.get_device_name(0)
            print('torch.npu.current_device():', lab2['current_device'])
            print('torch.npu.get_device_name(0):', lab2['device_name'])
    except Exception as exc:
        print(f'[FAIL] torch.npu 查询失败：{type(exc).__name__}: {exc}')

正常情况下，`torch.npu.is_available()` 应为 `True`，`torch.npu.device_count()` 至少为 1。`current_device()` 返回当前进程中的逻辑设备编号，该编号与 `npu-smi` 的驱动管理编号属于不同视图。

### 7. 执行最小 NPU 验证

在 `npu:0` 上创建一个 1 元素张量并等待 NPU 同步。这个操作会触发 `torch_npu` 后端注册、Runtime 初始化和设备内存分配，可以用来检查框架到设备的连通性。

In [ ]:
lab2['minimal_npu_ok'] = False
lab2['minimal_npu_error'] = None
if not lab2.get('npu_available') or lab2.get('torch_npu_device_count', 0) < 1:
    lab2['minimal_npu_error'] = '没有可用的 NPU，跳过最小分配。'
    print('[FAIL]', lab2['minimal_npu_error'])
else:
    try:
        probe_tensor = torch.empty(1, device='npu:0')
        torch.npu.synchronize()
        print('1 元素张量所在设备:', probe_tensor.device)
        lab2['minimal_npu_ok'] = str(probe_tensor.device).startswith('npu')
        del probe_tensor
        print('[PASS] 最小 NPU 分配与同步成功。')
    except Exception as exc:
        lab2['minimal_npu_error'] = f'{type(exc).__name__}: {exc}'
        print('[FAIL] 最小 NPU 初始化失败：', lab2['minimal_npu_error'])

张量设备应显示为 `npu:0`，同步完成后程序会打印 `[PASS]`。如果失败，保留首个异常单元的输出，并回到对应的软件层检查。

## 实验总结

本实验从 Python Kernel、驱动和 CANN 安装信息逐层检查到 ACL 与 `torch_npu`，最后通过一次最小张量分配确认 PyTorch 可以访问 NPU。各层输出对应不同的软件边界，排查时应先确认异常发生在哪一层。

## 实验扩展

1. `npu-smi` 顶部的 Version 属于哪一层？CANN Toolkit 版本从哪里读取？
2. 空闲环境中 AI Core 利用率为 0 时，还应查看哪个字段判断设备状态？
3. `torch_npu` 导入成功后，哪些结果共同确认 NPU 可以执行？
4. 哪两个环境变量通常指向 CANN Toolkit 根目录？哪个变量指向 OPP？
5. 在 `npu-smi` 成功且 `torch.npu.is_available()` 输出 `False` 时，应从哪两层开始定位？

运行下方单元读取参考答案。

In [ ]:
!cat answer/thought_questions.txt